# Smoke Test — All 6 Embedding Models

Run each cell in order. Each cell tests one model and is **independent** — if one fails, the others still run.

**Pass criteria per model**:
- Loads without error
- Output shape = `(1, expected_dim)`
- L2 norm ≈ 1.0 (unit vector)

In [1]:
import sys, os, time
from pathlib import Path
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from shared.embeddings import EmbeddingClient

CONFIG = ROOT / "models" / "config.yaml"
# device="cpu" for the smoke test: avoids silent MPS Metal JIT hang on first encode
client = EmbeddingClient(CONFIG, device="cpu")

PROBE_EN = "The court shall have jurisdiction over the matter."
PROBE_ZH = "法院對此事項具有管轄權。"

def check(label, model_id, probe, expected_dim):
    print(f"\n{'─'*60}")
    print(f"  Model : {label}")
    print(f"  ID    : {model_id}")
    print(f"  Probe : {probe[:50]}")
    print()

    print("  [1/2] Loading weights ...", end="", flush=True)
    t0 = time.perf_counter()
    try:
        client._get_model(model_id)
    except Exception as e:
        print(f" FAILED\n  ✗ {e}")
        return False
    print(f" {time.perf_counter()-t0:.1f}s  ✓")

    print("  [2/2] Encoding probe  ...", end="", flush=True)
    t0 = time.perf_counter()
    try:
        vecs = client.embed([probe], model_id, use_cache=False)
    except Exception as e:
        print(f" FAILED\n  ✗ {e}")
        return False
    print(f" {time.perf_counter()-t0:.1f}s  ✓")

    shape_ok = vecs.shape == (1, expected_dim)
    norm = float(np.linalg.norm(vecs[0]))
    norm_ok = abs(norm - 1.0) < 1e-4

    print()
    print(f"  shape  : {vecs.shape}   expected (1, {expected_dim})  {'✓' if shape_ok else '✗'}")
    print(f"  ‖v‖    : {norm:.6f}   expected ≈ 1.0              {'✓' if norm_ok else '✗'}")

    if shape_ok and norm_ok:
        print("\n  ✅ PASS")
        return True
    else:
        print("\n  ❌ FAIL")
        return False

results = {}
print("Client ready. Run the cells below — one per model.")
print(f"Config : {CONFIG}")
print(f"Device : {client._device}")

Client ready. Run the cells below — one per model.
Config : /Users/gpuzio/Desktop/CODE/THESIS/experiments/models/config.yaml
Device : cpu


---
## WEIRD — Slot 1: BGE-EN-large
`BAAI/bge-large-en-v1.5` · dim=1024 · architecture-control anchor

In [2]:
results["BGE-EN-large"] = check("BGE-EN-large", "BAAI/bge-large-en-v1.5", PROBE_EN, 1024)


────────────────────────────────────────────────────────────
  Model : BGE-EN-large
  ID    : BAAI/bge-large-en-v1.5
  Probe : The court shall have jurisdiction over the matter.

  [1/2] Loading weights ...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 2.5s  ✓
  [2/2] Encoding probe  ... 3.3s  ✓

  shape  : (1, 1024)   expected (1, 1024)  ✓
  ‖v‖    : 1.000000   expected ≈ 1.0              ✓

  ✅ PASS


## WEIRD — Slot 2: E5-large
`intfloat/e5-large-v2` · dim=1024 · STS-oriented

In [3]:
results["E5-large"] = check("E5-large", "intfloat/e5-large-v2", PROBE_EN, 1024)


────────────────────────────────────────────────────────────
  Model : E5-large
  ID    : intfloat/e5-large-v2
  Probe : The court shall have jurisdiction over the matter.

  [1/2] Loading weights ...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 2.5s  ✓
  [2/2] Encoding probe  ... 3.7s  ✓

  shape  : (1, 1024)   expected (1, 1024)  ✓
  ‖v‖    : 1.000000   expected ≈ 1.0              ✓

  ✅ PASS


## WEIRD — Slot 3: FreeLaw-EN
`freelawproject/modernbert-embed-base_finetune_512` · dim=768 · legally-informed

In [4]:
results["FreeLaw-EN"] = check("FreeLaw-EN", "freelawproject/modernbert-embed-base_finetune_512", PROBE_EN, 768)


────────────────────────────────────────────────────────────
  Model : FreeLaw-EN
  ID    : freelawproject/modernbert-embed-base_finetune_512
  Probe : The court shall have jurisdiction over the matter.

  [1/2] Loading weights ...

The `reference_compile` argument is deprecated and will be removed in `transformers v5.2.0`Use `torch.compile()` directly on the model instead.


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

 2.6s  ✓
  [2/2] Encoding probe  ... 0.8s  ✓

  shape  : (1, 768)   expected (1, 768)  ✓
  ‖v‖    : 1.000000   expected ≈ 1.0              ✓

  ✅ PASS


---
## Sinic — Slot 1: BGE-ZH-large
`BAAI/bge-large-zh-v1.5` · dim=1024 · architecture-control anchor

In [5]:
results["BGE-ZH-large"] = check("BGE-ZH-large", "BAAI/bge-large-zh-v1.5", PROBE_ZH, 1024)


────────────────────────────────────────────────────────────
  Model : BGE-ZH-large
  ID    : BAAI/bge-large-zh-v1.5
  Probe : 法院對此事項具有管轄權。

  [1/2] Loading weights ...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 2.6s  ✓
  [2/2] Encoding probe  ... 3.3s  ✓

  shape  : (1, 1024)   expected (1, 1024)  ✓
  ‖v‖    : 1.000000   expected ≈ 1.0              ✓

  ✅ PASS


## Sinic — Slot 2: Text2vec-large-ZH
`GanymedeNil/text2vec-large-chinese` · dim=1024 · STS-oriented (will download ~1.3GB)

In [6]:
results["Text2vec-large-ZH"] = check("Text2vec-large-ZH", "GanymedeNil/text2vec-large-chinese", PROBE_ZH, 1024)


────────────────────────────────────────────────────────────
  Model : Text2vec-large-ZH
  ID    : GanymedeNil/text2vec-large-chinese
  Probe : 法院對此事項具有管轄權。

  [1/2] Loading weights ...

No sentence-transformers model found with name GanymedeNil/text2vec-large-chinese. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: GanymedeNil/text2vec-large-chinese
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 1.4s  ✓
  [2/2] Encoding probe  ... 2.3s  ✓

  shape  : (1, 1024)   expected (1, 1024)  ✓
  ‖v‖    : 1.000000   expected ≈ 1.0              ✓

  ✅ PASS


## Sinic — Slot 3: Dmeta-ZH
`DMetaSoul/Dmeta-embedding-zh` · dim=768 · legally-informed (will download)

In [7]:
results["Dmeta-ZH"] = check("Dmeta-ZH", "DMetaSoul/Dmeta-embedding-zh", PROBE_ZH, 768)


────────────────────────────────────────────────────────────
  Model : Dmeta-ZH
  ID    : DMetaSoul/Dmeta-embedding-zh
  Probe : 法院對此事項具有管轄權。

  [1/2] Loading weights ...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

 2.5s  ✓
  [2/2] Encoding probe  ... 0.7s  ✓

  shape  : (1, 768)   expected (1, 768)  ✓
  ‖v‖    : 1.000000   expected ≈ 1.0              ✓

  ✅ PASS


---
## Summary

In [8]:
print(f"\n{'═'*60}")
print(f"  SMOKE TEST RESULTS")
print(f"{'═'*60}")
for label, ok in results.items():
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {label}")
passed = sum(results.values())
total = len(results)
print(f"{'─'*60}")
print(f"  {passed}/{total} models passed")
if passed == total:
    print("  Pipeline ready for Lens experiments.")
else:
    failed = [l for l, ok in results.items() if not ok]
    print(f"  Fix before proceeding: {failed}")


════════════════════════════════════════════════════════════
  SMOKE TEST RESULTS
════════════════════════════════════════════════════════════
  ✅  BGE-EN-large
  ✅  E5-large
  ✅  FreeLaw-EN
  ✅  BGE-ZH-large
  ✅  Text2vec-large-ZH
  ✅  Dmeta-ZH
────────────────────────────────────────────────────────────
  6/6 models passed
  Pipeline ready for Lens experiments.
